## 1. Project Objective

The objective of this project is to develop a Retrieval-Augmented Generation (RAG) system for answering questions from custom domain-specific documents.

The system combines semantic search with a Large Language Model. First, the document is divided into smaller chunks and converted into vector embeddings. These embeddings are stored in a FAISS vector database. When a user asks a question, the system retrieves the most relevant chunks and provides them as context to the LLM. The LLM then generates a grounded answer based only on the retrieved information.

The project demonstrates the complete RAG workflow:

Document Ingestion → Chunking → Embedding Creation → Vector Storage → Retrieval → Context Augmentation → Answer Generation

In [9]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

In [10]:

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import  RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# Loading the PDF file
## 2. Dataset / Document Description

The dataset used in this project is a 41-page PDF containing comprehensive Python programming notes.

The document covers multiple Python concepts, including:

- Python fundamentals
- Variables and data types
- Operators
- Conditional statements
- Loops
- Functions
- Object-Oriented Programming
- Classes and objects
- Inheritance
- Polymorphism
- Encapsulation
- Exception handling
- File handling
- Modules and packages
- Other important Python concepts

The document is domain-specific and contains structured explanations, making it suitable for demonstrating document-based question answering using RAG.

In [11]:
loader = PyPDFLoader("Rag/Python_Notes.pdf")

documents = loader.load()

print(f"Number of pages: {len(documents)}")

Number of pages: 41


# Chunking 

In [12]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)

In [13]:
print("No of chunk = ",len(chunks))

No of chunk =  118


In [14]:
print("For Ex First chunk : ", chunks[0])

For Ex First chunk :  page_content='1 
 
Unit - I  Introduction 
 
1.1 Introduction to Python   1.2 Features of 
python  
1.3 What Can I Do with Python?  1.4 Python 
Interpreter  
1.5 Data types, Variables, Comments, Operators, 
expressions; input, processing and output statements  
1.6 Control Structures: loops and decision  
Unit – II String Handling, Classes, Modules and 
Package  
2.1 Strings, String operations and String Slicing 
 2.2 Defining Classes  
2.3 Defining and calling functions passing arguments 
to functions  
2.4 Python and OOP – Inheritance, polymorphism 
 2.5 Modules – datetime, math  
2.6 Packages  
Unit – III - Exception Handling and Collections  
3.1  Exception in python  3.2 Exception roles  
3.3 Exception Handling  3.4 Collections in 
Python – List, Tuples, Dictionaries, Sets  
Unit – IV - GUI Programming and Database 
Connectivity Using Python  
4.1 Graphical User Interfaces 4.2  Using the tkinter 
Module  
4.3 Creating Label, Te xt, Button, info Dialog Boxes, 

# Experiment on Chunk size and overlap

| Configuration | Chunks | Precision | Context | Noise  | Overall  |
| ------------- | -----: | --------- | ------- | ------ | -------- |
| 500 / 100     |    231 | High      | Low     | Low    | Good     |
| 1000 / 200    |    118 | High      | High    | Low    | **Best** |
| 1500 / 300    |     84 | Medium    | High    | Medium | Good     |

The 1000-character chunk size with 200-character overlap performed best overall. The 500/100 configuration provided precise retrieval but sometimes lacked sufficient context, while the 1500/300 configuration provided broader context but introduced more irrelevant information. Therefore, the 1000/200 configuration offered the best balance between retrieval precision, contextual completeness, and minimal noise, making it the most suitable configuration for the final RAG pipeline.


# Embedding Creation
The `all-MiniLM-L6-v2` model was selected because it is lightweight, efficient, and suitable for semantic similarity search. It generates 384-dimensional embeddings and can run locally without requiring an external embedding API.

FAISS is used to store and efficiently search the vector embeddings of the document chunks. When a user submits a query, the query is converted into an embedding and compared with the stored document embeddings using vector similarity.

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(
    chunks,
    embeddings
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Retriever

In [17]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 8})

# Using Gemini LLM

In [18]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature = 0.2)

# Creating Prompt

In [19]:
prompt = PromptTemplate(
    template="""
    You are a helpful assistant.
    answer Onnly From the provided context.
    if the context is insufficient, just say "I don't know based on the provided context!".
    {context}
    Question: {question}
    """, input_variables = ['context', 'question']
)


In [20]:
def format_docs(retrieved_docs):
    context_text= "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text 

# Building Pipeline

In [21]:
parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough()
})

## Retrieval → Augmentation → Generation

In [22]:
result = parallel_chain.invoke(
    "What is the concept of inheritance?"
)

print("QUESTION:")
print(result["question"])

print("\nRETRIEVED CONTEXT:")
print(result["context"])

QUESTION:
What is the concept of inheritance?

RETRIEVED CONTEXT:
and the newly created class called as 
Derived/Child/Sub class. 
-When inheritance is applied, it shows following 
effects: 
 i) Copy of data member is transferred from 
base to derived class & 
 ii) permission given to child class to call base 
class function without object. 
- Inheritance gives biggest advantage of reusability. 
- In Python, following types of inheritances are 
supported: 
1) Single  2) Multi-level 3) Multiple  
4) Hierarchical 5) Hybrid 
1) Single – In this inheritance, only one base class and 
only one derived class is involved: 
    
 
 
 
 
Syntax : Class DerivedClassName(BaseClsName): 
Base 
class 
Derived 
class

25 
 
         self.m=int(input("Enter IInd Value=")) 
class C(A,B): 
    def display(self): 
        self.getIn1() 
        self.getIn2() 
        print("Addition=",self.n+self.m) 
========= to execute =========== 
>>> ob=C() 
>>> ob.display() 
Enter Ist Value=12 
Enter IInd Value=23 
(

In [23]:
parser = StrOutputParser()

In [24]:
main_chain = parallel_chain | prompt | llm | parser

# Asking Questions

In [25]:
question = "can you tell me Types of inheritance with one example."

answer = main_chain.invoke(question)

print(answer)

Based on the provided context, the types of inheritances supported in Python are:

1. **Single**
2. **Multi-level**
3. **Multiple**
4. **Hierarchical**
5. **Hybrid**

### Example of Single Inheritance:
In this type of inheritance, only one base class and only one derived class are involved.

```python
class A: 
    n=0; 
    def display(self): 
        self.n=input("Enter any Value=") 
        print("in Base Class n=",self.n) 

class B(A): 
    def show(self): 
        self.display() 
        print("in Derived Class n=",self.n) 
```


##  Observations

- The RAG system successfully loaded and processed the 41-page Python notes PDF.
- RecursiveCharacterTextSplitter divided the document into smaller overlapping chunks, making semantic retrieval more effective.
- The `all-MiniLM-L6-v2` embedding model provided an efficient local solution for converting document chunks into vector representations.
- FAISS enabled fast similarity-based retrieval of relevant document chunks.
- The similarity retriever successfully retrieved Python concepts related to user queries.
- A lower temperature value of 0.2 helped produce more consistent and factual responses.
- The RAG pipeline generated answers based on the retrieved context rather than relying solely on the LLM's general knowledge.
- Increasing the number of retrieved chunks can provide more context but may also introduce irrelevant information.
- The choice of chunk size and overlap affects the quality and relevance of retrieved information.

##  Conclusion

A Retrieval-Augmented Generation system was successfully developed to answer questions from a 41-page Python programming document. The system followed the complete RAG workflow: document loading, text chunking, embedding generation, FAISS vector storage, similarity-based retrieval, context augmentation, and LLM-based answer generation.

The use of the `all-MiniLM-L6-v2` embedding model provided an efficient local approach for semantic search, while FAISS enabled fast retrieval of relevant document chunks. Gemini 2.5 Flash was used to generate responses based on the retrieved context.

The system demonstrated that RAG can effectively provide domain-specific answers from custom documents and reduce the need for the LLM to rely only on its internal knowledge. Experiments with chunk size, overlap, and the number of retrieved documents showed that retrieval configuration has a significant impact on answer quality.

Overall, the project successfully demonstrates how Retrieval-Augmented Generation can be used to build a document-based question-answering system.